# 🧪 Laboratorio de estrategias SOL (operaciones ≤ 48 h)

Prueba estrategias sacadas de papers **sin trampas**: walk-forward (los parámetros se eligen con el pasado
y se prueban en meses que el optimizador no vio), costes reales (comisión + slippage + funding) y
comparación contra **buy & hold** y una **regla simple**.

Una estrategia solo "PASA" si fuera de muestra: gana dinero, bate a los benchmarks en Sharpe, su
Deflated Sharpe Ratio es ≥ 0.95 (no se explica por suerte tras probar cientos de combinaciones) y
**sigue funcionando en los últimos 6 meses**.

Archivos: `data.py` (descarga y caché), `lab.py` (backtest y walk-forward), `strategies.py` (papers).
Para añadir un paper: escribe su función en `strategies.py` y añádela a `STRATEGIES`.

In [ ]:
# Solo la primera vez en el PC
%pip install -q pandas numpy scipy requests matplotlib

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data, lab, strategies
for m in (data, lab, strategies):
    importlib.reload(m)  # para recoger cambios en los .py sin reiniciar el kernel

from data import load_dataset
from lab import backtest, buy_and_hold, deflated_sharpe, metrics, walk_forward, TAKER, MAKER, Execution
from strategies import STRATEGIES, simple_rule_entries

SOURCE = 'binance'      # 'binance' (2023 → hoy) o 'hyperliquid' (~7 meses, mercado del bot)
START = '2023-01'
TRAIN_DAYS, TEST_DAYS = 90, 30   # en hyperliquid usa 60 / 14

# Ejecución: TAKER (a mercado) o órdenes límite, p. ej. Execution('maker', entry_offset=0.001)
EXECUTION = TAKER

## 1. Datos

In [ ]:
df = load_dataset(SOURCE, START)
print(f"{len(df)} velas de 1 h: {df.index[0]} → {df.index[-1]}")
df.tail()

## 2. Walk-forward de todas las estrategias

In [ ]:
runs = []
for s in STRATEGIES:
    if any(c not in df.columns for c in s.requires):
        print(f"⏭️  {s.name}: faltan {s.requires}")
        continue
    print(f"🔬 {s.name} ({len(s.param_sets())} combinaciones)")
    runs.append(walk_forward(df, s, TRAIN_DAYS, TEST_DAYS, execution=EXECUTION))

all_trials = [sr for r in runs for sr in r['trial_sharpes']]
for r in runs:
    r['metrics']['psr'], r['metrics']['dsr'] = deflated_sharpe(r['result'].returns, all_trials)
print(f"Configuraciones probadas: {len(all_trials)}")

In [ ]:
first, end = runs[0]['period']
oos = df[df.index >= first]
bh = buy_and_hold(oos)
simple = backtest(oos, simple_rule_entries(oos), 24)

rows = {'[bench] buy_and_hold': metrics(bh), '[bench] regla_simple': metrics(simple.returns, simple.trades)}
for r in runs:
    rows[r['strategy'].name] = r['metrics']
table = pd.DataFrame(rows).T
table[['retorno_total', 'sharpe', 'max_drawdown', 'operaciones', 'aciertos', 'media_por_op', 'psr', 'dsr', 'ventanas_positivas']]

## 3. Curvas de capital fuera de muestra (escala log)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
(1 + bh).cumprod().plot(ax=ax, color='black', lw=2, label='buy & hold')
(1 + simple.returns).cumprod().plot(ax=ax, color='gray', ls='--', label='regla simple')
for r in runs:
    (1 + r['result'].returns).cumprod().plot(ax=ax, label=r['strategy'].name)
ax.set_yscale('log'); ax.set_ylabel('capital (1 = inicio)'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'Fuera de muestra: {EXECUTION.label()} + funding')
plt.show()

## 4. ¿Es robusta o suerte? Sharpe de cada combinación en todo el periodo

Una estrategia real funciona en **muchas** combinaciones vecinas. Si solo gana una celda aislada, es suerte.

In [ ]:
NAME = 'ruptura_volatilidad'   # cambia por la estrategia a inspeccionar
s = next(x for x in STRATEGIES if x.name == NAME)
res = []
for p in s.param_sets():
    ret = backtest(df, s.entries(df, df if s.needs_fit else None, **p), p['hold'], EXECUTION).returns
    res.append({**p, 'sharpe': ret.mean() / ret.std() * np.sqrt(lab.BARS_PER_YEAR)})
res = pd.DataFrame(res)
keys = [k for k in s.grid if k != 'hold']
res.pivot_table(index=keys, columns='hold', values='sharpe').round(2).style.background_gradient(cmap='RdYlGn', vmin=-1.5, vmax=1.5)

## 5. Detalle de una estrategia: por año, largos vs cortos, ventanas

In [ ]:
r = next(x for x in runs if x['strategy'].name == NAME)
t = r['result'].trades.copy()
t['año'] = t['entry_time'].dt.year
display(t.groupby('side')['net_return'].agg(ops='count', media='mean', total='sum'))
display(t.groupby('año')['net_return'].agg(ops='count', media='mean', total='sum'))
r['windows'].tail(12)

## 6. Plantilla: añade tu propio paper

1. Copia esta función a `strategies.py`, cámbiale la lógica y añade un `Strategy(...)` a `STRATEGIES`.
2. Vuelve a ejecutar desde la celda 2. Si no pasa, anota en `docs/ESTADO_BOT.md` que está descartada y por qué.

In [ ]:
def mi_estrategia_entries(df, train, lookback, hold):
    # Ejemplo: largo si el cierre supera la media de `lookback` horas, corto si queda por debajo.
    # Usa SOLO datos hasta la vela actual (nada de shift(-n)).
    ma = df['close'].rolling(lookback).mean()
    return np.sign(df['close'] - ma).fillna(0)

prueba = lab.Strategy(name='mi_estrategia', reference='Autor (año) "Título"', idea='...',
                      grid={'lookback': [24, 48, 96], 'hold': [12, 24, 48]}, entries=mi_estrategia_entries)
r = walk_forward(df, prueba, TRAIN_DAYS, TEST_DAYS, execution=EXECUTION)
r['metrics']